In [3]:
import pandas as pd
import re

# =========================
# LOAD DATA
# =========================

df = pd.read_csv("output_full/full_parsed.csv")

# =========================
# NORMALIZATION FUNCTION
# =========================

# alias dictionary
ALIASES = {
    # languages
    "js": "javascript",
    "ts": "typescript",
    "golang": "go",

    # frameworks / libraries
    "reactjs": "react",
    "react.js": "react",
    "vuejs": "vue",
    "vue.js": "vue",
    "nextjs": "next.js",
    "nodejs": "node.js",
    "expressjs": "express.js",

    # databases
    "postgres": "postgresql",
    "mongo": "mongodb",

    # cloud / devops
    "aws cloud": "aws",
    "amazon web services": "aws",
    "google cloud platform": "gcp",

    # ai / data
    "machine learning": "ml",
    "deep learning": "dl",
    "artificial intelligence": "ai",
}

# filler phrases thường xuất hiện trong JD
FILLER_PHRASES = [
    "experience with",
    "experience in",
    "knowledge of",
    "hands-on experience with",
    "hands-on experience in",
    "familiar with",
    "proficiency in",
    "understanding of",
    "ability to use",
    "working knowledge of",
    "good understanding of",
    "strong knowledge of",
]

def normalize_skill(skill):

    if pd.isna(skill):
        return None

    # =========================
    # BASIC CLEANING
    # =========================

    skill = str(skill).lower().strip()

    # remove extra spaces
    skill = re.sub(r"\s+", " ", skill)

    # remove surrounding punctuation
    skill = re.sub(r"^[^\w]+|[^\w]+$", "", skill)

    # =========================
    # REMOVE FILLER PHRASES
    # =========================

    for phrase in FILLER_PHRASES:
        if skill.startswith(phrase):
            skill = skill.replace(phrase, "").strip()

    # =========================
    # REMOVE COMMON NOISE WORDS
    # =========================

    noise_words = [
        "skills",
        "skill",
        "experience",
        "knowledge",
        "expertise",
        "framework",
        "library",
        "tool",
        "platform",
    ]

    pattern = r"\b(" + "|".join(noise_words) + r")\b"
    skill = re.sub(pattern, "", skill)

    # clean spaces again
    skill = re.sub(r"\s+", " ", skill).strip()

    # =========================
    # STANDARDIZE SYMBOLS
    # =========================

    skill = skill.replace(" / ", "/")
    skill = skill.replace(" & ", " and ")

    # =========================
    # APPLY ALIAS MAPPING
    # =========================

    skill = ALIASES.get(skill, skill)

    # =========================
    # FINAL CLEAN
    # =========================

    skill = skill.strip()

    return skill


# =========================
# APPLY NORMALIZATION
# =========================

df["normalized_skill"] = df["skill_name"].apply(normalize_skill)

In [4]:
import pandas as pd
import os
import re

# =========================
# CLEAN CATEGORY
# =========================

df["category"] = (
    df["category"]
    .astype(str)
    .str.strip()
)

# =========================
# OUTPUT FOLDER
# =========================

output_dir = "category_skill_statistics"
os.makedirs(output_dir, exist_ok=True)

# =========================
# CREATE STATISTICS
# =========================

skill_stats = (
    df.groupby(["category", "normalized_skill"])
      .size()
      .reset_index(name="count")
      .sort_values(
          by=["category", "count", "normalized_skill"],
          ascending=[True, False, True]
      )
)

# =========================
# SAVE ONE FILE PER CATEGORY
# =========================

for category in skill_stats["category"].unique():

    category_df = (
        skill_stats[
            skill_stats["category"] == category
        ]
        .reset_index(drop=True)
    )

    # safe filename
    safe_category = re.sub(
        r'[\\/*?:"<>|]',
        "_",
        category
    )

    file_path = os.path.join(
        output_dir,
        f"{safe_category}.csv"
    )

    category_df.to_csv(file_path, index=False)

    print(
        f"Saved: {file_path} "
        f"({len(category_df):,} skills)"
    )

# =========================
# SUMMARY
# =========================

print("\nDone.")
print(f"Total categories: {skill_stats['category'].nunique():,}")
print(f"Output folder: {output_dir}")

Saved: category_skill_statistics\AI_ML_Data.csv (955 skills)
Saved: category_skill_statistics\Data Engineering & Analytics.csv (533 skills)
Saved: category_skill_statistics\Database.csv (356 skills)
Saved: category_skill_statistics\Design & UX.csv (474 skills)
Saved: category_skill_statistics\Domain Knowledge.csv (2,467 skills)
Saved: category_skill_statistics\Embedded & Firmware.csv (406 skills)
Saved: category_skill_statistics\Engineering Concepts & Methodologies.csv (2,326 skills)
Saved: category_skill_statistics\Framework _ Library.csv (701 skills)
Saved: category_skill_statistics\IT Support & Hardware.csv (348 skills)
Saved: category_skill_statistics\Infrastructure & DevOps.csv (1,932 skills)
Saved: category_skill_statistics\Other.csv (956 skills)
Saved: category_skill_statistics\Programming Language.csv (183 skills)
Saved: category_skill_statistics\Soft Skill.csv (1,288 skills)
Saved: category_skill_statistics\Testing & QA.csv (913 skills)
Saved: category_skill_statistics\Tool & 

In [3]:
import pandas as pd

pd.set_option("display.max_rows", None)          # Hiện tất cả dòng
pd.set_option("display.max_columns", None)       # Hiện tất cả cột
pd.set_option("display.max_colwidth", None)      # Hiện toàn bộ nội dung trong ô
pd.set_option("display.width", None)             # Không giới hạn chiều rộng
pd.set_option("display.expand_frame_repr", False)

In [4]:
import pandas as pd

jobs   = pd.read_parquet("../../data/interim/02-skill_extracted/jobs.parquet")
skills = pd.read_parquet("../../data/interim/02-skill_extracted/skills.parquet")

no_skill_ids = set(jobs["job_id"]) - set(skills["job_id"])
no_skill = jobs[jobs["job_id"].isin(no_skill_ids)].copy()

print(f"Jobs không có skill extraction: {len(no_skill)}")
print(f"Breakdown theo source:\n{no_skill['source'].value_counts().to_string()}")
print()

display(
    no_skill[["job_id", "source", "standardized_title", "url",
              "requirement", "platform_required_skills", "platform_preferred_skills","requirement"]]
    .reset_index(drop=True)
)

Jobs không có skill extraction: 13
Breakdown theo source:
source
topcv    13



,job_id,source,standardized_title,url,requirement,platform_required_skills,platform_preferred_skills,requirement
0,115,topcv,IT Support,https://www.topcv.vn/viec-lam/nhan-vien-it-phan-cung-phan-mem-luong-tu-8-18-trieu-thang-khong-kinh-nghiem-duoc-dao-tao/2097246.html,"Trình độ:Trung cấp, Cao đẳng, Đại học tốt nghiệp các ngành Cơ khí/Kĩ thuật ứng dụng, công nghệ thông tin, Điện điện tử, Phần mềm lập trình Kinh nghiệm : chưa có kinh nghiệm và từ 1 năm kinh nghiệm, nếu chưa có kinh nghiệm sẽ được đào tạo (có nhận học viên) Nhanh nhẹn, siêng năng Trung thực trong công việc Có tinh thần trách nhiệm","['Cài Đặt Phần Mềm', 'Sửa Chữa Kỹ Thuật', 'sửa chưa bảo dưỡng']","['Cài Đặt Phần Mềm', 'Sửa Chữa Kỹ Thuật']","Trình độ:Trung cấp, Cao đẳng, Đại học tốt nghiệp các ngành Cơ khí/Kĩ thuật ứng dụng, công nghệ thông tin, Điện điện tử, Phần mềm lập trình Kinh nghiệm : chưa có kinh nghiệm và từ 1 năm kinh nghiệm, nếu chưa có kinh nghiệm sẽ được đào tạo (có nhận học viên) Nhanh nhẹn, siêng năng Trung thực trong công việc Có tinh thần trách nhiệm"
1,290,topcv,IT Support,https://www.topcv.vn/viec-lam/nhan-vien-it-linh-vuc-fb/1164345.html,"• Giới tính: Nam, độ tuổi 20- 28 • Trình độ: tốt nghiệp từ trung cấp trở lên chuyên ngành Công nghệ thông tin, điện tử viễn thông.... • Kinh nghiệm làm việc hoặc thực tập từ 6 tháng trở lên • Có khả năng di chuyển, nhiệt tình, ham học hỏi, chịu khó.....",[],[],"• Giới tính: Nam, độ tuổi 20- 28 • Trình độ: tốt nghiệp từ trung cấp trở lên chuyên ngành Công nghệ thông tin, điện tử viễn thông.... • Kinh nghiệm làm việc hoặc thực tập từ 6 tháng trở lên • Có khả năng di chuyển, nhiệt tình, ham học hỏi, chịu khó....."
2,346,topcv,IT Support,https://www.topcv.vn/viec-lam/nhan-vien-it-phan-cung/2092555.html,Tốt nghiệp từ cao đẳng các ngành có liên quan. Nam độ tuổi từ 25-30 Ưu tiên ứng viên có kinh nghiệm từ 1 năm trở lên làm các công việc tương tự,"['Kiến Thức Về Phần Cứng Máy Tính', 'Kiến Thức Về Mạng Máy Tính', 'Khả Năng Chẩn Đoán Và Khắc Phục Sự Cố', 'Khả Năng Sửa Chữa Phần Cứng']","['Kiến Thức Về Bảo Mật Máy Tính', 'Kinh Nghiệm Làm Việc Với Các Hệ Thống Ảo Hóa', 'Kinh Nghiệm Làm Việc Với Các Công Cụ Chẩn Đoán Phần Cứng']",Tốt nghiệp từ cao đẳng các ngành có liên quan. Nam độ tuổi từ 25-30 Ưu tiên ứng viên có kinh nghiệm từ 1 năm trở lên làm các công việc tương tự
3,724,topcv,IT Support,https://www.topcv.vn/viec-lam/nhan-vien-it/2032985.html,"tốt nghiệp hệ trung cấp, cao đẳng, đại học công nghệ thông tin. có kinh nghiệm từ 2 năm có đủ sức khoẻ và tinh thần trách nhiệm trong công việc.",[],[],"tốt nghiệp hệ trung cấp, cao đẳng, đại học công nghệ thông tin. có kinh nghiệm từ 2 năm có đủ sức khoẻ và tinh thần trách nhiệm trong công việc."
4,813,topcv,IT Support,https://www.topcv.vn/viec-lam/nhan-vien-ho-tro-ky-thuat-may-tinh/2089469.html,"• Độ tuổi: 19-30 tuổi • Không yêu cầu kinh nghiệm (được đào tạo từ đầu) • Cẩn thận, trung thực, có tinh thần trách nhiệm cao trong công việc • Yêu thích công việc kỹ thuật",[],[],"• Độ tuổi: 19-30 tuổi • Không yêu cầu kinh nghiệm (được đào tạo từ đầu) • Cẩn thận, trung thực, có tinh thần trách nhiệm cao trong công việc • Yêu thích công việc kỹ thuật"
5,876,topcv,IT Instructor,https://www.topcv.vn/viec-lam/giao-vien-part-time-sinh-vien-nganh-cong-nghe-thong-tin-ky-thuat-dien-tu-tu-dong-hoa-hai-phong/1363980.html,"Đang là sinh viên năm 2,3,4 đang theo học các chuyên ngành Công nghệ thông tin, Điện tử, Tự động hoá, Mỹ thuật,Toán,…. của các trường cao đẳng, đại học liên quan. Tốt nghiệp Đại học chuyên ngành Sư phạm, khoa công nghệ thông tin. Là giảng viên đã và đang giảng dạy tại các trường cấp 1, cấp 2 bộ môn Tin học, Công nghệ thông tin. Hoặc đang làm việc tại các công ty Truyền thông, Công nghệ thông tin.","['Kỹ năng sư phạm', 'Quản Lý Lớp Học', 'Giao tiếp tốt', 'Khả Năng Truyền Đạt Kiến Thức', 'Kiến Thức Chuyên Môn Về Lĩnh Vực Giảng Dạy (cntt, Thiết Kế, Điện Tử Viễn Thông, Truyền Thông Đa Phương Tiện Hoặc Sư Phạm Kỹ Thuật)']",[],"Đang là sinh viên năm 2,3,4 đang theo học các chuyên ng